# SparseWalker streaming-native training speed audit

This notebook speeds up the PR17 streaming-native ML-1M trainer **without changing model semantics**. It keeps persistent K=8 local state, two causal temporal layers, full chronological histories, and TBPTT detach at chunk boundaries.

The speed branch adds: one pinned H2D batch transfer, no per-chunk CUDA→CPU synchronization, grouped backward calls across already-detached TBPTT chunks, fixed 64-event training chunk shapes, and an optional `torch.compile` local-Walker chunk.

Before timing, it gates correctness with (1) grouped-backward parameter-update parity, (2) optimized-eager vs PR17 local state parity, and (3) compiled vs eager concept-ID/mass/hidden parity. Compile failures fall back cleanly to eager.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, json, runpy, torch
from pathlib import Path

REPO='/content/Sparsewalker'
BRANCH='agent/streaming-training-speed-v3'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO], check=True)
sys.path.insert(0, f'{REPO}/experiments')
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU', torch.cuda.get_device_name(0))
print('torch', torch.__version__, 'bf16', torch.cuda.is_bf16_supported())
print('BRANCH', BRANCH)


## 1. Short speed sweep

This runs the same optimized trainer from identical initial weights at batch 32/64/128, eager and compiled-local. It uses only 24 batches per cell so we can choose a configuration before spending a full training run.


In [ ]:
SCRIPT=f'{REPO}/experiments/run_ml1m_walker_streaming_speed.py'
sys.argv=[
    SCRIPT,
    '--benchmark-only',
    '--benchmark-batches','24',
    '--benchmark-batch-sizes','32,64,128',
    '--backward-group-chunks','4',
    '--benchmark-compile',
]
runpy.run_path(SCRIPT, run_name='__main__')


In [ ]:
import pandas as pd
p=Path('/content/drive/MyDrive/sparsewalker_streaming_speed/ml1m/seed42/speed_sweep.json')
rows=json.loads(p.read_text())
flat=[]
for r in rows:
    flat.append({
        'status':r.get('status'),
        'batch_size':r.get('batch_size'),
        'compiled_local':r.get('compiled_local'),
        'positions_per_s':r.get('positions_per_s'),
        'seconds':r.get('seconds'),
        'chunks':r.get('chunks'),
        'backward_calls':r.get('backward_calls'),
        'error':r.get('error'),
    })
df=pd.DataFrame(flat)
display(df.sort_values('positions_per_s', ascending=False, na_position='last'))

# For the first quality-bearing run, cap the automatic choice at batch<=64.
# Batch 128 remains in the sweep as a throughput ceiling but changes optimizer
# step frequency more aggressively and should get its own quality check.
safe=[r for r in rows if r.get('status')=='OK' and int(r['batch_size'])<=64]
if not safe: raise RuntimeError('No passing speed cell at batch<=64')
best=max(safe, key=lambda r: r['positions_per_s'])
SELECT_BATCH=int(best['batch_size'])
SELECT_COMPILE=bool(best['compiled_local'])
print('SELECTED', {'batch_size':SELECT_BATCH, 'compile_local':SELECT_COMPILE, 'positions_per_s':best['positions_per_s']})


## 2. Full streaming-native training

Run the fastest passing configuration at batch ≤64 for 30 epochs. Evaluation remains eager/reference semantics and reports full persistent-history NDCG@10 plus the same weights reset to the last 200 events.


In [ ]:
argv=[
    SCRIPT,
    '--epochs','30',
    '--eval-every','5',
    '--batch-size',str(SELECT_BATCH),
    '--eval-batch-size','64',
    '--chunk-size','64',
    '--memory-size','512',
    '--backward-group-chunks','4',
]
argv += ['--compile-local'] if SELECT_COMPILE else ['--no-compile-local']
sys.argv=argv
runpy.run_path(SCRIPT, run_name='__main__')


In [ ]:
root=Path('/content/drive/MyDrive/sparsewalker_streaming_speed/ml1m/seed42')
for name in ['speed_config.json','history.json','result.json']:
    q=root/name
    if q.exists():
        print('\n###', name)
        obj=json.loads(q.read_text())
        if name=='history.json':
            display(pd.DataFrame(obj))
        else:
            print(json.dumps(obj, indent=2))


## SWG note

This notebook deliberately does **not** put SWG/HNSW into the training loop. The current temporal SWG kernel is a serving/search primitive, while temporal keys change on every optimizer step. After the dense-trained streaming model passes quality, the next serving experiment should retrieve semantic candidates with SWG, union them with recent/temporal anchors, then exact-rerank the small candidate set using the streaming model's true `q·k + relative-lag bias` score.
